# Évaluer l'agent et choisir son modèle : banc d'évaluation, juge, coût, analyse d'erreurs

Le fil rouge de la formation : l'agent du service client de Biclou & Cie, qui lit le message d'un client, appelle les outils du magasin (la fiche d'une commande, le stock, les règles du magasin, un ticket, le transfert à un conseiller), puis répond selon la charte. Le chapitre 4 a affiné son modèle, Qwen3-1.7B, par QLoRA, et l'a mesuré appel par appel ; le chapitre 5 l'a servi derrière une API compatible avec celle d'OpenAI et a rendu la boucle de l'agent robuste. Ce TP construit le banc d'évaluation qui décide du modèle à mettre en production. Il compare trois moteurs sous la même boucle :

- `base` : le modèle de base, Qwen3, servi par llama.cpp ;
- `specialise` : le même modèle avec l'adaptateur LoRA du chapitre 4, servi avec le gabarit de l'entraînement ;
- `generaliste` : un modèle généraliste par API.

1. Le modèle, appel par appel
2. L'agent de bout en bout
3. Un LLM juge pour les réponses
4. Coût et latence
5. Analyse d'erreurs et recommandation

Ce notebook tourne avec ou sans GPU : llama.cpp sert les modèles sur le GPU s'il y en a un (*Exécution > Modifier le type d'exécution > GPU T4*), sur le processeur sinon. Il mesure un échantillon du jeu figé, `N_ECHANTILLON` demandes (cellule de configuration). Sur GPU, il sert Qwen3-1.7B, le modèle du cours ; sans GPU, Qwen3-0.6B (`MODELE`), et tourne en 40 minutes environ sur les deux processeurs de Colab (une estimation). Les exercices sont les mêmes. Un échantillon suffit à écrire le banc et à le vérifier, pas à départager les modèles : l'exercice 5 dit pourquoi. Le banc complet, sur les 139 demandes, se fait sur GPU.

Le généraliste par API demande une clé, comme au chapitre 5 : le secret Colab `OPENAI_API_KEY` (la clé dans la barre de gauche), et, en option, `OPENAI_BASE_URL` pour un autre fournisseur compatible et `GENERALISTE_MODELE` pour le nom du modèle. Sans clé, le notebook sert ici Qwen3-4B-Instruct-2507, qui écrit quelques tokens par seconde sur deux processeurs, trop lent pour passer le banc : il n'y joue que le juge de l'exercice 3, sur quelques réponses, et les autres exercices comparent le modèle de base et le modèle spécialisé.

## Données, modèles et outils

Les données de l'agent, dans le dossier `agent/` du jeu `bike-shop-support` : `eval.jsonl`, le jeu d'évaluation figé (139 demandes, avec leurs appels attendus, les résultats de ces appels et une réponse de référence), sur lequel ce TP mesure et rien n'est réglé ; `commandes.csv`, `stock.csv` et `charte.md`, ce que lisent les outils. `agent-sft-validation.jsonl`, le jeu de validation du chapitre 4 : la consigne de l'agent et les schémas des outils, tels que le modèle les a vus à l'entraînement. `qwen3_training.jinja` : le gabarit de Qwen3 avec lequel `trl` 1.18 a entraîné l'adaptateur. Et llama.cpp, à l'étiquette `b12359` : son dépôt, pour ses scripts de conversion au format GGUF, et ses programmes compilés pour Linux (`llama-server`) : pour CUDA 12.8 sur GPU, pour le processeur sans GPU.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/eval.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/commandes.csv
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/stock.csv
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/charte.md
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/lab-results/agent-sft-validation.jsonl
!wget -q https://raw.githubusercontent.com/huggingface/trl/v1.18.0/trl/chat_templates/qwen3_training.jinja
!git clone -q --depth 1 --branch b12359 https://github.com/ggml-org/llama.cpp
import torch

GPU = torch.cuda.is_available()
# Les programmes de llama.cpp : pour CUDA 12.8 sur GPU, pour le processeur sans GPU.
if GPU:
  !wget -q -O llama.tar.gz https://github.com/ggml-org/llama.cpp/releases/download/b12359/llama-b12359-bin-ubuntu-cuda-12.8-x64.tar.gz
else:
  !wget -q -O llama.tar.gz https://github.com/ggml-org/llama.cpp/releases/download/b12359/llama-b12359-bin-ubuntu-x64.tar.gz
!tar -xzf llama.tar.gz
%pip install -q "mcp>=2,<3" openai sentencepiece ./llama.cpp/gguf-py

In [ ]:
import json
import math
import os
import re
import subprocess
import sys
import time
import urllib.request
from dataclasses import dataclass

import jsonschema
import openai
import pandas as pd
import requests
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from mcp import Client, StdioServerParameters
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, ValidationError

# Le modèle de l'agent : sur GPU, Qwen3-1.7B, celui du chapitre 4 ; sans GPU,
# Qwen3-0.6B, qui génère environ trois fois plus vite (une estimation), avec
# l'adaptateur de référence de ce modèle-là.
MODELE = "Qwen/Qwen3-1.7B" if GPU else "Qwen/Qwen3-0.6B"
TAILLE = MODELE.split("-")[-1].lower()  # "1.7b" ou "0.6b"
# Les demandes du jeu figé que mesure le banc, tirées selon leur nombre d'appels
# attendus. Les 139 demandent cinq heures environ avec Qwen3-0.6B sur deux
# processeurs (une estimation) : elles se mesurent sur GPU.
N_ECHANTILLON = 12
# Les exigences de l'équipe, fixées avant le banc : le taux de réussite de bout en
# bout au moins, la latence p95 d'une demande au plus, en secondes. L'exercice 5
# choisit parmi les moteurs qui les tiennent.
SEUIL_REUSSITE = 0.8
P95_MAX = 30
# L'étiquette de llama.cpp clonée et téléchargée ci-dessus, et ses programmes.
LLAMA_CPP = "b12359"
BINAIRES = f"llama-{LLAMA_CPP}"
# Les processeurs dont dispose le notebook, 2 sur Colab : llama.cpp, lui, compterait
# les cœurs physiques de la machine.
CPUS = len(os.sched_getaffinity(0))
# Sur GPU, llama-server y place toutes les couches du modèle.
OPTIONS_GPU = ["--n-gpu-layers", "99"] if GPU else []
print(MODELE, "sur GPU" if GPU else f"sans GPU, sur {CPUS} processeurs")


def lance(commande: list[str]) -> str:
  """Lance une commande et renvoie sa sortie ; si elle échoue, lève une erreur."""
  resultat = subprocess.run(commande, capture_output=True, text=True)
  sortie = resultat.stdout + resultat.stderr
  if resultat.returncode != 0:
    raise RuntimeError(f"{commande[1]} a échoué :\n{sortie[-3000:]}")
  return sortie


def secret(nom: str) -> str | None:
  """Un secret de Colab (la clé dans la barre de gauche), sinon une variable
  d'environnement ; None s'il n'existe pas."""
  try:
    from google.colab import userdata
  except ImportError:
    return os.environ.get(nom)
  try:
    return userdata.get(nom)
  except Exception:  # pas de tel secret, ou pas d'accès
    return None

L'adaptateur et les fichiers GGUF, comme au chapitre 5 : votre adaptateur, si vous l'avez publié à la fin du chapitre 4 (le secret Colab `HF_TOKEN`, l'étiquette `v1.0`) et qu'il a été entraîné sur `MODELE`, sinon celui de référence, de [shuuchuu/models](https://github.com/shuuchuu/models) ; puis le modèle de base converti en GGUF, quantifié en Q8_0, et l'adaptateur en un GGUF à part, en 16 bits.

In [ ]:
ADAPTATEUR = "agent-sft-lora"
REFERENCE = f"https://github.com/shuuchuu/models/raw/refs/heads/main/agent-sft-lora-qwen3-{TAILLE}"


def adaptateur_du_hub() -> str | None:
  """Télécharge dans ADAPTATEUR la version v1.0 de votre adaptateur, s'il existe
  pour MODELE."""
  jeton = secret("HF_TOKEN")
  if not jeton:
    return None
  api = HfApi(token=jeton)
  depot = f"{api.whoami()['name']}/agent-sft-lora"
  if not api.repo_exists(depot):
    return None
  if "v1.0" not in [etiquette.name for etiquette in api.list_repo_refs(depot).tags]:
    print(f"{depot} n'a pas d'étiquette v1.0 : l'adaptateur de référence le remplace")
    return None
  with open(api.hf_hub_download(depot, "adapter_config.json", revision="v1.0")) as f:
    base = json.load(f)["base_model_name_or_path"]
  if base != MODELE:
    print(f"{depot} est entraîné sur {base}, pas sur {MODELE} : l'adaptateur de")
    print("référence le remplace")
    return None
  api.snapshot_download(depot, revision="v1.0", local_dir=ADAPTATEUR)
  return depot


depot = adaptateur_du_hub()
if depot is None:
  os.makedirs(ADAPTATEUR, exist_ok=True)
  for fichier in ["adapter_config.json", "adapter_model.safetensors"]:
    urllib.request.urlretrieve(f"{REFERENCE}/{fichier}", f"{ADAPTATEUR}/{fichier}")
print("Adaptateur :", depot or REFERENCE)

os.makedirs("gguf", exist_ok=True)
GGUF_BASE = f"gguf/qwen3-{TAILLE}-q8_0.gguf"
GGUF_LORA = "gguf/agent-sft-lora-f16.gguf"
DOSSIER_MODELE = snapshot_download(
  MODELE, allow_patterns=["*.json", "*.safetensors", "*.txt"]
)
lance(
  [
    sys.executable,
    "llama.cpp/convert_hf_to_gguf.py",
    DOSSIER_MODELE,
    "--outfile",
    GGUF_BASE,
    "--outtype",
    "q8_0",
  ]
)
lance(
  [
    sys.executable,
    "llama.cpp/convert_lora_to_gguf.py",
    ADAPTATEUR,
    "--base",
    DOSSIER_MODELE,
    "--outfile",
    GGUF_LORA,
    "--outtype",
    "f16",
  ]
)
print("Conversions faites :", os.listdir("gguf"))

Le système du magasin et son serveur MCP, repris tels quels des chapitres 2 et 5 : `biclou.py`, ses données et les fonctions des outils ; `serveur_biclou.py`, le serveur MCP qui les déclare, en stdio.

In [ ]:
%%writefile biclou.py
"""Le système de Biclou & Cie : ses données, et ce que font les outils de l'agent."""

import re
import unicodedata

import pandas as pd

commandes = pd.read_csv("commandes.csv", dtype=str, keep_default_na=False)
commandes = commandes.set_index("numero")
stock = pd.read_csv("stock.csv", keep_default_na=False)
with open("charte.md", encoding="utf-8") as f:
  charte = f.read()

CATEGORIES = [
  "information",
  "livraison",
  "reparation",
  "retour",
  "location",
  "facturation",
  "compte",
]
PRIORITES = ["basse", "normale", "haute"]
RAISONS = ["securite", "colis_perdu", "paiement", "accident_location", "vol_location"]
MODELES = list(dict.fromkeys(stock["modele"]))
TAILLES = list(dict.fromkeys(stock["taille"]))
MAGASINS = list(dict.fromkeys(stock["magasin"]))
RUBRIQUES = {
  "Retours": "retours",
  "Livraison": "livraison",
  "Garantie": "garantie",
  "Atelier": "atelier",
  "Location": "location",
  "Facturation": "facturation",
  "Compte": "compte",
  "Magasins": "magasins",
  "Tailles de cadre": "tailles",
}


def lit_politique(charte: str) -> dict[str, list[str]]:
  """Les phrases de chaque rubrique de la partie « Politique » de la charte."""
  texte = charte.replace("\xa0", " ").split("## Politique", 1)[1]
  puces = re.findall(r"^- \*\*(.+?)\*\* : (.+?)(?=^- |\Z)", texte, re.M | re.S)
  return {
    RUBRIQUES[titre]: re.split(
      r"(?<=\.) (?=[A-ZÀ-Ý])", f"{titre} : {' '.join(corps.split())}"
    )
    for titre, corps in puces
  }


PASSAGES = lit_politique(charte)
THEMES = list(PASSAGES)


def racines(texte: str) -> set[str]:
  """Les mots de 4 lettres ou plus, comparés par leurs 5 premières lettres."""
  texte = unicodedata.normalize("NFKD", texte.lower())
  texte = "".join(c for c in texte if not unicodedata.combining(c))
  return {mot[:5] for mot in re.findall(r"[a-z]+", texte) if len(mot) >= 4}


def chercher_commande(numero: str) -> dict:
  """La fiche de la commande, sans ses champs vides, ou une erreur."""
  if not re.fullmatch(r"CMD-\d{6}", numero):
    return {"erreur": "numéro invalide : CMD- suivi de 6 chiffres"}
  if numero not in commandes.index:
    return {"erreur": f"commande introuvable : {numero}"}
  fiche = {"numero": numero, **commandes.loc[numero].to_dict()}
  fiche = {cle: valeur for cle, valeur in fiche.items() if valeur != ""}
  fiche["montant"] = float(fiche["montant"])
  fiche["paiements"] = int(fiche["paiements"])
  return fiche


def verifier_stock(modele: str, taille: str, magasin: str) -> dict:
  """La quantité en stock et, en rupture, la date du prochain réassort."""
  ligne = stock[
    (stock["modele"] == modele)
    & (stock["taille"] == taille)
    & (stock["magasin"] == magasin)
  ]
  if ligne.empty:
    return {"erreur": "modèle, taille ou magasin inconnu"}
  quantite = int(ligne["quantite"].iloc[0])
  resultat = dict(modele=modele, taille=taille, magasin=magasin, quantite=quantite)
  if quantite == 0:
    resultat["reassort"] = ligne["reassort"].iloc[0]
  return resultat


def chercher_politique(theme: str, question: str) -> dict:
  """Les trois passages de la rubrique les plus proches de la question."""
  if theme not in PASSAGES:
    return {"erreur": f"rubrique inconnue : {theme}"}
  mots = racines(question)
  tries = sorted(
    enumerate(PASSAGES[theme]), key=lambda p: (-len(mots & racines(p[1])), p[0])
  )
  return {"theme": theme, "passages": [passage for _, passage in tries[:3]]}


compteurs = {"ticket": 30000, "dossier": 5000}
EQUIPES = {
  "securite": ("atelier", "aujourd'hui"),
  "colis_perdu": ("livraison", "sous 24 h"),
  "paiement": ("comptabilite", "sous 24 h"),
  "accident_location": ("location", "sous 24 h"),
  "vol_location": ("location", "sous 24 h"),
}


def creer_ticket(categorie: str, priorite: str, resume: str) -> dict:
  """Le numéro du ticket créé."""
  if categorie not in CATEGORIES or priorite not in PRIORITES:
    return {"erreur": "catégorie ou priorité inconnue"}
  compteurs["ticket"] += 1
  return {"ticket": f"T-{compteurs['ticket']}", "statut": "ouvert"}


def escalader(raison: str, resume: str) -> dict:
  """Le numéro du dossier transmis et l'équipe qui le prend en charge."""
  if raison not in EQUIPES:
    return {"erreur": f"raison inconnue : {raison}"}
  compteurs["dossier"] += 1
  equipe, recontact = EQUIPES[raison]
  return {
    "dossier": f"ESC-{compteurs['dossier']}",
    "equipe": equipe,
    "recontact": recontact,
  }

In [ ]:
%%writefile serveur_biclou.py
"""Le serveur MCP des outils de Biclou & Cie, en stdio."""

from typing import Annotated, Literal

import biclou
from mcp.server.mcpserver import MCPServer
from pydantic import Field

serveur = MCPServer("biclou", log_level="WARNING")


@serveur.tool()
def chercher_commande(
  numero: Annotated[
    str, Field(description="le numéro de commande, CMD- suivi de 6 chiffres")
  ],
) -> dict:
  """Cherche une commande dans le système de commandes."""
  return biclou.chercher_commande(numero)


@serveur.tool()
def verifier_stock(
  modele: Annotated[
    Literal[tuple(biclou.MODELES)], Field(description="le modèle de vélo")
  ],
  taille: Annotated[
    Literal[tuple(biclou.TAILLES)], Field(description="la taille de cadre")
  ],
  magasin: Annotated[
    Literal[tuple(biclou.MAGASINS)], Field(description="la ville du magasin")
  ],
) -> dict:
  """Vérifie le stock d'un modèle de vélo dans un magasin."""
  return biclou.verifier_stock(modele, taille, magasin)


@serveur.tool()
def chercher_politique(
  theme: Annotated[
    Literal[tuple(biclou.THEMES)],
    Field(description="la rubrique des règles où chercher"),
  ],
  question: Annotated[str, Field(description="la question, en quelques mots")],
) -> dict:
  """Cherche dans les règles du magasin les passages qui répondent à une question."""
  return biclou.chercher_politique(theme, question)


@serveur.tool()
def creer_ticket(
  categorie: Annotated[
    Literal[tuple(biclou.CATEGORIES)],
    Field(description="la catégorie de la demande"),
  ],
  priorite: Annotated[
    Literal[tuple(biclou.PRIORITES)], Field(description="la priorité de la demande")
  ],
  resume: Annotated[str, Field(description="la demande en une phrase")],
) -> dict:
  """Crée un ticket pour une demande que l'équipe doit traiter."""
  return biclou.creer_ticket(categorie, priorite, resume)


@serveur.tool()
def escalader(
  raison: Annotated[
    Literal[tuple(biclou.RAISONS)], Field(description="la raison du transfert")
  ],
  resume: Annotated[str, Field(description="la situation en une phrase")],
) -> dict:
  """Transmet la demande à un conseiller, pour un problème de sécurité ou d'argent."""
  return biclou.escalader(raison, resume)


if __name__ == "__main__":
  serveur.run()

Le client MCP et ce dont la boucle de l'agent a besoin, du chapitre 5 : `SERVEUR`, les paramètres de lancement du serveur MCP ; `execute(mcp, appel)`, le résultat d'un appel par le serveur ; `SCHEMAS`, les schémas des arguments des outils, tels que le serveur les déclare ; la consigne de l'agent, `CONSIGNES_AGENT`, et les schémas des outils de l'entraînement, `OUTILS`, ceux que l'agent envoie au modèle ; `conversation(message)`, `nettoie`, la réponse de repli `REPLI` et le budget `MAX_ETAPES`.

In [ ]:
eval_fige = pd.read_json("eval.jsonl", lines=True)
nombre_appels = eval_fige["appels_attendus"].str.len()
with open("agent-sft-validation.jsonl", encoding="utf-8") as f:
  validation = [json.loads(ligne) for ligne in f]
# La consigne et les schémas des outils, tels que le modèle les a vus à l'entraînement.
CONSIGNES_AGENT = validation[0]["messages"][0]["content"]
OUTILS_ENTRAINEMENT = json.loads(validation[0]["tools"])


def sans_retour(outil: dict) -> dict:
  """Un schéma d'outil sans la clé `return`, que l'API d'OpenAI n'a pas."""
  fonction = {cle: v for cle, v in outil["function"].items() if cle != "return"}
  return dict(type=outil["type"], function=fonction)


OUTILS = [sans_retour(o) for o in OUTILS_ENTRAINEMENT]  # ce que l'agent envoie

SERVEUR = StdioServerParameters(command=sys.executable, args=["serveur_biclou.py"])


async def execute(mcp, appel: dict) -> dict:
  """Le résultat d'un appel par le serveur, ou l'erreur que l'agent recevra."""
  resultat = await mcp.call_tool(appel["name"], appel["arguments"])
  texte = resultat.content[0].text
  return {"erreur": texte} if resultat.is_error else json.loads(texte)


async with Client(SERVEUR) as mcp:
  SCHEMAS = {o.name: o.input_schema for o in (await mcp.list_tools()).tools}


def conversation(message: str, systeme: str = CONSIGNES_AGENT) -> list[dict]:
  return [dict(role="system", content=systeme), dict(role="user", content=message)]


def nettoie(texte: str) -> str:
  """Sans les espaces de fin de ligne que le modèle laisse parfois."""
  return "\n".join(ligne.rstrip() for ligne in texte.strip().splitlines())


SANS_PENSEE = {"chat_template_kwargs": {"enable_thinking": False}}
# Les générations au plus par message du client : deux appels, une relance, la réponse.
MAX_ETAPES = 4
REPLI = """Bonjour,

Merci pour votre message. Nous n'avons pas pu traiter votre demande automatiquement : un conseiller de notre équipe la reprend dès maintenant et vous répondra par écrit dans les meilleurs délais, avec les informations de votre dossier. Vous n'avez rien à faire d'ici là.

Bonne journée,
L'équipe Biclou & Cie"""
print(len(eval_fige), "demandes dans le jeu figé ; les outils :", list(SCHEMAS))

Les deux serveurs locaux, comme le chapitre 5 les a laissés (ses exercices 1 et 2) : `demarre_serveur`, `arrete` ; `qwen3-base` sur le port 8080, le modèle de base avec son gabarit ; `biclou-agent` sur le port 8081, avec l'adaptateur et le gabarit de l'entraînement. Sur GPU, `OPTIONS_GPU` (`--n-gpu-layers 99`) y place toutes les couches du modèle. `--cache-ram` borne à 1 Gio le cache des prompts en mémoire vive de chaque serveur (8 Gio par défaut), où llama.cpp range les prompts qu'une autre requête remplace dans le *slot*, pour les reprendre plus tard : le banc envoie des centaines de prompts, et trois serveurs doivent tenir dans la mémoire de Colab.

Un changement : un `Moteur` porte aussi les options de ses requêtes (`options`). Un modèle par API n'accepte pas forcément celles d'un serveur local : un modèle de raisonnement refuse `temperature=0` et `max_tokens`, un fournisseur, les champs qu'il ne connaît pas, comme `chat_template_kwargs`.

In [ ]:
@dataclass
class Moteur:
  """Un modèle servi : le client de son serveur, le nom du modèle, les options de ses
  requêtes."""

  client: OpenAI
  modele: str
  options: dict


SERVEURS = {}  # les processus llama-server, par nom


def arrete(nom: str) -> None:
  """Arrête le serveur `nom` s'il tourne, et attend qu'il ait fini."""
  if nom in SERVEURS:
    serveur = SERVEURS.pop(nom)
    serveur.terminate()
    serveur.wait()


def demarre_serveur(
  nom: str, port: int, options: list[str] = (), gguf: str = GGUF_BASE
) -> subprocess.Popen:
  """Lance llama-server sur `gguf`, sous l'alias `nom`, et attend qu'il soit prêt."""
  arrete(nom)
  journal = open(f"{nom}.log", "w")
  commande = [
    f"{BINAIRES}/llama-server",
    "--model",
    gguf,
    "--port",
    str(port),
    "--alias",
    nom,
    "--threads",
    str(CPUS),
    "--parallel",
    "1",
    "--ctx-size",
    "4096",
    "--cache-ram",
    "1024",
    *OPTIONS_GPU,
    *options,
  ]
  serveur = subprocess.Popen(commande, stdout=journal, stderr=subprocess.STDOUT)
  SERVEURS[nom] = serveur
  while True:
    try:
      if requests.get(f"http://localhost:{port}/health").status_code == 200:
        return serveur
    except requests.ConnectionError:
      pass
    if serveur.poll() is not None:
      raise RuntimeError(f"llama-server s'est arrêté : lisez {nom}.log")
    time.sleep(1)


demarre_serveur("qwen3-base", 8080)
demarre_serveur(
  "biclou-agent",
  8081,
  ["--lora", GGUF_LORA, "--chat-template-file", "qwen3_training.jinja"],
)
# Sans échantillonnage, 200 tokens au plus, la pensée éteinte, comme au chapitre 5.
LOCAL = dict(temperature=0, max_tokens=200, extra_body=SANS_PENSEE)
BASE = Moteur(
  OpenAI(base_url="http://localhost:8080/v1", api_key="aucune"), "qwen3-base", LOCAL
)
SPECIALISE = Moteur(
  OpenAI(base_url="http://localhost:8081/v1", api_key="aucune"), "biclou-agent", LOCAL
)
print("Serveurs prêts :", list(SERVEURS))

Le généraliste, comme au chapitre 5 (son exercice 5) : un modèle par API si vous avez une clé, sinon Qwen3-4B-Instruct-2507, quantifié en Q4_K_M, servi ici même (2,5 Go à télécharger), qui ne fait alors que le juge de l'exercice 3, sur `N_JUGE_LOCAL` réponses par moteur. `MOTEURS` : les moteurs que comparent les exercices 1, 2, 4 et 5.

In [ ]:
cle = secret("OPENAI_API_KEY")
PAR_API = bool(cle)
if PAR_API:
  client = OpenAI(api_key=cle, base_url=secret("OPENAI_BASE_URL"), timeout=120)
  # Les options du fournisseur : aucune. gpt-5.4-mini est un modèle de raisonnement,
  # et ceux d'OpenAI refusent max_tokens (max_completion_tokens à la place) et,
  # selon leur réglage, temperature=0. Pour un modèle qui l'accepte, temperature=0
  # rend ses réponses, et le juge, reproductibles.
  GENERALISTE = Moteur(client, secret("GENERALISTE_MODELE") or "gpt-5.4-mini", {})
else:
  chemin = hf_hub_download(
    "unsloth/Qwen3-4B-Instruct-2507-GGUF",
    "Qwen3-4B-Instruct-2507-Q4_K_M.gguf",
    revision="a06e946bb6b655725eafa393f4a9745d460374c9",
  )
  demarre_serveur("generaliste", 8082, gguf=chemin)
  client = OpenAI(base_url="http://localhost:8082/v1", api_key="aucune")
  GENERALISTE = Moteur(client, "generaliste", LOCAL)
N_JUGE_LOCAL = 2  # sans clé, les réponses jugées par moteur
MOTEURS = {"base": BASE, "specialise": SPECIALISE}
if PAR_API:
  MOTEURS["generaliste"] = GENERALISTE
print("Généraliste :", GENERALISTE.modele, "par API" if PAR_API else "servi ici")
print("Moteurs comparés :", list(MOTEURS))

Ce que les exercices 2 à 4 du chapitre 5 ont écrit, et dont le banc se sert : `exacts`, `meme_appel` (le même outil, avec les mêmes arguments vérifiables), `lit_appel`, `appel_chat`, `compte_tokens`, `trouves` (les appels attendus retrouvés parmi les obtenus, dans n'importe quel ordre), puis `verifie` et `agent_robuste`. Un seul changement dans `agent_robuste` : ses requêtes prennent les options du moteur.

In [ ]:
ARGUMENTS_LIBRES = {"question", "resume"}  # du texte, qu'on ne compare pas mot à mot


def exacts(appel: dict) -> dict:
  """Les arguments vérifiables d'un appel : sans les arguments libres."""
  arguments = appel["arguments"] if isinstance(appel["arguments"], dict) else {}
  return {a: v for a, v in arguments.items() if a not in ARGUMENTS_LIBRES}


def meme_appel(obtenu: dict | None, attendu: dict) -> bool:
  """Le même outil, avec les mêmes arguments vérifiables."""
  return (
    obtenu is not None
    and obtenu["name"] == attendu["name"]
    and exacts(obtenu) == exacts(attendu)
  )


def lit_appel(appel_outil) -> dict:
  """Un appel de `tool_calls` en dictionnaire ; ses arguments restent du texte quand
  ils ne sont pas un JSON valide."""
  try:
    arguments = json.loads(appel_outil.function.arguments)
  except json.JSONDecodeError:
    arguments = appel_outil.function.arguments
  return dict(name=appel_outil.function.name, arguments=arguments)


def appel_chat(appel: dict) -> dict:
  """Un appel au format des conversations : ses arguments en texte JSON."""
  arguments = json.dumps(appel["arguments"], ensure_ascii=False)
  return dict(type="function", function=dict(name=appel["name"], arguments=arguments))


def compte_tokens(trajet: dict, usage) -> None:
  """Ajoute au trajet les tokens d'une requête : le prompt, dont ceux du cache, et
  les tokens générés."""
  trajet["tokens_prompt"] = trajet.get("tokens_prompt", 0) + usage.prompt_tokens
  details = usage.prompt_tokens_details
  caches = details.cached_tokens if details and details.cached_tokens else 0
  trajet["tokens_caches"] = trajet.get("tokens_caches", 0) + caches
  trajet["tokens_generes"] = trajet.get("tokens_generes", 0) + usage.completion_tokens


def trouves(attendus: list[dict], obtenus: list[dict]) -> int:
  """Les appels attendus qu'on retrouve parmi les obtenus (meme_appel)."""
  restants = list(obtenus)
  n = 0
  for attendu in attendus:
    for obtenu in restants:
      if meme_appel(obtenu, attendu):
        restants.remove(obtenu)
        n += 1
        break
  return n

In [ ]:
SCHEMAS_STRICTS = {
  nom: schema | {"additionalProperties": False} for nom, schema in SCHEMAS.items()
}
IDEMPOTENTS = {"chercher_commande", "verifier_stock", "chercher_politique"}
RELANCES_MAX = 2  # erreurs renvoyées au modèle au plus, par message du client


def verifie(appel: dict, messages: list[dict]) -> list[str]:
  """Les problèmes d'un appel, avant de l'exécuter ; vide s'il est valide."""
  if appel["name"] not in SCHEMAS_STRICTS:
    return [f"outil inconnu : {appel['name']}"]
  arguments = appel["arguments"]
  if not isinstance(arguments, dict):
    return [f"arguments illisibles, un objet JSON attendu : {arguments}"]
  validateur = jsonschema.Draft202012Validator(SCHEMAS_STRICTS[appel["name"]])
  problemes = [erreur.message for erreur in validateur.iter_errors(arguments)]
  # La règle métier : un numéro de commande vient du client ou d'un outil qui a réussi.
  lu = " ".join(
    m["content"]
    for m in messages
    if m["role"] == "user" or (m["role"] == "tool" and '"erreur"' not in m["content"])
  )
  numero = arguments.get("numero")
  if isinstance(numero, str):
    if not re.fullmatch(r"CMD-\d{6}", numero):
      problemes.append(f"numéro de commande mal formé : {numero}")
    elif numero not in lu:
      problemes.append(f"numéro de commande absent de la conversation : {numero}")
  return problemes


def refus(problemes: list[str]) -> dict:
  return {"erreur": "appel refusé : " + " ; ".join(problemes)}


async def agent_robuste(moteur, mcp, messages, bavard=False) -> dict:
  """La boucle de l'agent, qui valide chaque appel avant de l'exécuter, renvoie au
  modèle les refus et les erreurs des outils, ne refait jamais une écriture, et passe
  la main à un conseiller quand elle ne peut pas conclure."""
  trajet = dict(
    appels=[],
    format_lisible=True,
    generations=0,
    reponse=REPLI,
    refus=[],
    relances=0,
    transfert=None,
  )
  faits = {}  # les écritures exécutées, et leur résultat
  for _ in range(MAX_ETAPES):
    try:
      reponse = moteur.client.chat.completions.create(
        model=moteur.modele, messages=messages, tools=OUTILS, **moteur.options
      )
    except openai.APIError as erreur:  # après les relances du client
      trajet["transfert"] = f"serveur du modèle : {type(erreur).__name__}"
      break
    trajet["generations"] += 1
    compte_tokens(trajet, reponse.usage)
    message = reponse.choices[0].message
    if not message.tool_calls:  # la réponse
      trajet["reponse"] = nettoie(message.content or "") or REPLI
      break
    messages.append(message.model_dump(exclude_none=True))
    for appel_outil in message.tool_calls:
      appel = lit_appel(appel_outil)
      problemes = verifie(appel, messages)
      if problemes:  # refusé avant toute exécution
        observation = refus(problemes)
        trajet["refus"] += problemes
      elif appel["name"] in faits:  # une écriture déjà faite : pas deux fois
        observation = faits[appel["name"]]
      else:
        observation = None
        for _ in range(2 if appel["name"] in IDEMPOTENTS else 1):
          try:
            observation = await execute(mcp, appel)
            break
          except Exception:  # l'outil n'a pas répondu
            pass
        if observation is None:  # une écriture a peut-être eu lieu
          trajet["transfert"] = f"{appel['name']} sans réponse"
          break
        trajet["appels"].append(appel)
        if appel["name"] not in IDEMPOTENTS:
          faits[appel["name"]] = observation
      if "erreur" in observation:
        trajet["relances"] += 1
      messages.append(
        dict(
          role="tool",
          tool_call_id=appel_outil.id,
          content=json.dumps(observation, ensure_ascii=False),
        )
      )
      if bavard:
        print("Action :", appel["name"], appel["arguments"], "->", observation)
    if trajet["transfert"] is None and trajet["relances"] > RELANCES_MAX:
      trajet["transfert"] = "trop de relances"
    if trajet["transfert"]:
      break
  else:
    trajet["transfert"] = "budget d'étapes épuisé"
  if trajet["transfert"]:
    trajet["reponse"] = REPLI
  messages.append(dict(role="assistant", content=trajet["reponse"]))
  if bavard:
    print("Transfert :", trajet["transfert"], "| Réponse :", trajet["reponse"][:80])
  return trajet

L'échantillon du banc : `N_ECHANTILLON` demandes du jeu figé, tirées au hasard dans chaque groupe de demandes au même nombre d'appels attendus (aucun, un ou deux), en proportion de sa taille. Un tirage stratifié garde dans l'échantillon chaque sorte de demande ; un tirage au hasard sur tout le jeu pourrait, sur douze demandes, n'en prendre aucune à deux appels. La graine est fixe : chaque moteur passe les mêmes demandes.

In [ ]:
echantillon = eval_fige.groupby(nombre_appels, group_keys=False).sample(
  frac=N_ECHANTILLON / len(eval_fige), random_state=0
)
print(len(echantillon), "demandes ; par nombre d'appels attendus :")
print(echantillon["appels_attendus"].str.len().value_counts().sort_index().to_dict())
print(echantillon["scenario"].value_counts().to_dict())

## 1. Le modèle, appel par appel

### 1.1 Les appels d'outils, étape par étape

La mesure du chapitre 4, par l'API cette fois, et pour chaque moteur. La cellule ci-dessous en reprend les définitions : `etapes(demande)`, l'historique correct de chaque étape, reconstruit à partir des appels attendus et de leurs résultats (avec, en plus, les identifiants d'appel qu'une API exige) ; `valide`, celle du chapitre 4, les problèmes d'un appel face au schéma de son outil tel que le modèle l'a vu à l'entraînement (`OUTILS`) : plus stricte que le `valide` de l'exercice 7 du chapitre 2, qui suit le schéma que déclare le serveur MCP (`SCHEMAS`), elle refuse aussi un argument inconnu et un `numero` mal formé, que ce schéma accepte ; `erreur_de(attendus, trajet)`, la classe d'une sortie, de `format` à `correct` ; `NIVEAUX` et `TAUX`, les quatre taux emboîtés, de `format_lisible` à `correct`. `options_etape(moteur)` : les options d'une étape, 96 tokens au plus pour un moteur servi ici, comme au chapitre 4.

Écrivez :

- `sortie_etape(moteur, messages)`, la sortie du moteur pour une étape, sous la forme du trajet que lit `erreur_de`, `{"appels", "format_lisible"}` : la suite de `messages` par l'API, avec les outils `OUTILS` et `options_etape(moteur)`, ses appels lus par `lit_appel`. La sortie est illisible quand des arguments ne sont pas un JSON valide, quand le texte garde un bloc `<tool_call>` que le serveur n'a pas su lire, ou quand le serveur renvoie une erreur (`openai.APIError`) ;
- `pas_a_pas(nom, moteur, demandes)`, le détail par demande, comme `note` au chapitre 4 : le nom du moteur, l'`id`, le nombre d'appels attendus, les quatre taux, la classe de la première étape fausse (`erreur`, `"correct"` sinon), son rang (`etape`), et la classe de chaque étape (`erreurs`), que l'exercice 5 relira.

Mesurez chaque moteur de `MOTEURS` sur l'`echantillon` et rangez les détails, tous moteurs confondus, dans `pas`. Affichez les quatre taux par moteur, les classes d'erreur par moteur, et le taux `correct` par nombre d'appels attendus : sur les demandes qui n'en attendent aucun, un appel est une erreur (`appel_inutile`). Où chaque moteur décroche-t-il ?

**Documentation utile**

- [`pandas.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)
- [`DataFrame.pivot_table`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html)

In [ ]:
def etapes(demande: dict) -> list[tuple[list[dict], list[dict]]]:
  """Les étapes d'une demande : pour chaque appel attendu, la conversation qui le
  précède et cet appel ; puis la conversation entière, où aucun appel n'est attendu."""
  messages = conversation(demande["message"])
  liste = []
  paires = zip(demande["appels_attendus"], demande["resultats_outils"])
  for i, (appel, resultat) in enumerate(paires):
    liste.append((messages, [appel]))
    identifiant = f"appel-{i}"  # une API exige l'identifiant de chaque appel
    messages = messages + [
      dict(
        role="assistant",
        content="",
        tool_calls=[appel_chat(appel) | {"id": identifiant}],
      ),
      dict(
        role="tool",
        tool_call_id=identifiant,
        content=json.dumps(resultat, ensure_ascii=False),
      ),
    ]
  liste.append((messages, []))
  return liste


PARAMETRES = {o["function"]["name"]: o["function"]["parameters"] for o in OUTILS}


def valide(appel: dict) -> list[str]:
  """Les problèmes d'un appel face au schéma de son outil, [] s'il est valide."""
  schema = PARAMETRES.get(appel["name"])
  if schema is None:
    return [f"outil inconnu : {appel['name']}"]
  arguments = appel["arguments"]
  if not isinstance(arguments, dict):
    return ["les arguments ne sont pas un objet"]
  proprietes = schema["properties"]
  problemes = [f"{a} manquant" for a in schema["required"] if a not in arguments]
  for a, valeur in arguments.items():
    if a not in proprietes:
      problemes.append(f"{a} inconnu")
    elif not isinstance(valeur, str):
      problemes.append(f"{a} n'est pas une chaîne")
    elif valeur not in proprietes[a].get("enum", [valeur]):
      problemes.append(f"{a} hors de la liste : {valeur}")
  numero = arguments.get("numero")
  if isinstance(numero, str) and not re.fullmatch(r"CMD-\d{6}", numero):
    problemes.append(f"numero mal formé : {numero}")
  return problemes


def erreur_de(attendus: list[dict], trajet: dict) -> str:
  obtenus = trajet["appels"]
  noms_attendus = [a["name"] for a in attendus]
  noms_obtenus = [a["name"] for a in obtenus]
  if not trajet["format_lisible"]:
    return "format"
  if noms_obtenus != noms_attendus:
    if noms_obtenus == noms_attendus[: len(noms_obtenus)]:
      return "appel_manquant"
    if noms_attendus == noms_obtenus[: len(noms_attendus)]:
      return "appel_inutile"
    return "mauvais_outil"
  if any(valide(a) for a in obtenus):
    return "arguments_invalides"
  if any(exacts(o) != exacts(a) for o, a in zip(obtenus, attendus)):
    return "arguments_faux"
  return "correct"


# Le niveau qu'atteint une étape : chaque taux compte les demandes dont toutes les
# étapes le dépassent.
NIVEAUX = {
  "format": 0,
  "appel_manquant": 1,
  "appel_inutile": 1,
  "mauvais_outil": 1,
  "arguments_invalides": 2,
  "arguments_faux": 3,
  "correct": 4,
}
TAUX = ["format_lisible", "bons_outils", "arguments_valides", "correct"]


def options_etape(moteur: Moteur) -> dict:
  """Les options d'une étape : 96 tokens au plus pour un moteur servi ici ; un modèle
  par API garde les siennes."""
  courtes = {"max_tokens": 96} if "max_tokens" in moteur.options else {}
  return moteur.options | courtes


print(sum(len(etapes(d)) for d in echantillon.to_dict("records")), "étapes par moteur")

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Une étape : `moteur.client.chat.completions.create(model=moteur.modele, messages=messages, tools=OUTILS, **options_etape(moteur))`, dans un `try` qui rattrape `openai.APIError` ; `.choices[0].message`, son texte `.content` (parfois `None`) et ses appels `.tool_calls` (`None` sans appel)
- `isinstance(a["arguments"], dict)` est faux pour des arguments que `lit_appel` n'a pas pu lire
- Le détail d'une demande, comme au chapitre 4 : `niveau = min(NIVEAUX[e] for e in erreurs)`, puis `{taux: niveau > i for i, taux in enumerate(TAUX)}` ; `fausses = [i for i, e in enumerate(erreurs) if e != "correct"]`
- `pd.concat([...], ignore_index=True)` : les détails des moteurs, l'un sous l'autre ; `pas.groupby("moteur")[TAUX].mean()`, `pd.crosstab(pas["erreur"], pas["moteur"])`, `pas.pivot_table(index="appels_attendus", columns="moteur", values="correct")`
- Sur deux processeurs, une étape prend de quelques secondes à une demi-minute avec Qwen3-0.6B : quelques minutes par moteur

```python
def sortie_etape(moteur: Moteur, messages: list[dict]) -> dict:
  try:
    reponse = moteur.client.chat.completions.create(...)
  except openai.APIError:
    return dict(appels=[], format_lisible=False)
  message = reponse.choices[0].message
  appels = [lit_appel(a) for a in message.tool_calls or []]
  lisible = all(isinstance(a["arguments"], dict) for a in appels)
  lisible = lisible and "<tool_call>" not in (message.content or "")
  return dict(appels=appels, format_lisible=lisible)


def pas_a_pas(nom: str, moteur: Moteur, demandes: pd.DataFrame) -> pd.DataFrame:
  lignes = []
  for demande in demandes.to_dict("records"):
    erreurs = [
      erreur_de(attendus, sortie_etape(moteur, messages))
      for messages, attendus in etapes(demande)
    ]
    niveau = ...
    fausses = ...
    lignes.append(
      dict(
        moteur=nom,
        id=demande["id"],
        appels_attendus=len(demande["appels_attendus"]),
        **{taux: niveau > i for i, taux in enumerate(TAUX)},
        erreur=erreurs[fausses[0]] if fausses else "correct",
        etape=...,
        erreurs=erreurs,
      )
    )
  return pd.DataFrame(lignes)


pas = pd.concat(
  [pas_a_pas(nom, moteur, echantillon) for nom, moteur in MOTEURS.items()],
  ignore_index=True,
)
print(pas.groupby("moteur")[TAUX].mean().round(2))
print(pd.crosstab(...))
pas.pivot_table(...).round(2)
```

#### Solution

In [ ]:
def sortie_etape(moteur: Moteur, messages: list[dict]) -> dict:
  """La sortie du moteur pour une étape, en trajet : ses appels, et s'ils sont
  lisibles."""
  try:
    reponse = moteur.client.chat.completions.create(
      model=moteur.modele, messages=messages, tools=OUTILS, **options_etape(moteur)
    )
  except openai.APIError:  # le serveur n'a pas su lire la sortie, ou ne répond pas
    return dict(appels=[], format_lisible=False)
  message = reponse.choices[0].message
  appels = [lit_appel(a) for a in message.tool_calls or []]
  lisible = all(isinstance(a["arguments"], dict) for a in appels)
  lisible = lisible and "<tool_call>" not in (message.content or "")
  return dict(appels=appels, format_lisible=lisible)


def pas_a_pas(nom: str, moteur: Moteur, demandes: pd.DataFrame) -> pd.DataFrame:
  """Le détail par demande, chaque étape jugée sur l'historique correct."""
  lignes = []
  for demande in demandes.to_dict("records"):
    erreurs = [
      erreur_de(attendus, sortie_etape(moteur, messages))
      for messages, attendus in etapes(demande)
    ]
    niveau = min(NIVEAUX[e] for e in erreurs)
    fausses = [i for i, e in enumerate(erreurs) if e != "correct"]
    lignes.append(
      dict(
        moteur=nom,
        id=demande["id"],
        appels_attendus=len(demande["appels_attendus"]),
        **{taux: niveau > i for i, taux in enumerate(TAUX)},
        erreur=erreurs[fausses[0]] if fausses else "correct",
        etape=fausses[0] if fausses else None,
        erreurs=erreurs,
      )
    )
  return pd.DataFrame(lignes)


debut = time.time()
pas = pd.concat(
  [pas_a_pas(nom, moteur, echantillon) for nom, moteur in MOTEURS.items()],
  ignore_index=True,
)
print(f"{time.time() - debut:.0f} s\n")
print(pas.groupby("moteur")[TAUX].mean().round(2), "\n")
print(pd.crosstab(pas["erreur"], pas["moteur"]), "\n")
pas.pivot_table(index="appels_attendus", columns="moteur", values="correct").round(2)

# Le serveur lit lui-même les blocs <tool_call> : une sortie illisible devient rare,
# sans disparaître (un bloc mal fermé reste dans le texte). Regardez à quel taux chaque
# moteur décroche, et la classe de sa première erreur. Un modèle qui répond sans
# appeler fait des appels manquants, qui passent inaperçus sur les demandes sans appel
# attendu ; un modèle affiné pour appeler ses outils peut en appeler un sans raison,
# sur ces mêmes demandes. Un modèle par API fait parfois deux appels d'un coup : la
# mesure, un appel par étape, le compte appel_inutile ; l'exercice 2 le juge sur le
# trajet entier.

### 1.2 Non-régression : ce que l'affinage a coûté

L'affinage apprend une tâche ; il peut en désapprendre d'autres, ou les déformer : un modèle entraîné à appeler des outils et à signer « L'équipe Biclou & Cie » peut se mettre à le faire partout, c'est la fuite du format. `GENERALES` (ci-dessous) : huit questions générales, du calcul, des connaissances, de la langue, chacune avec le motif d'une réponse juste. Les deux dernières sont posées avec les outils de l'agent, sans sa consigne, et n'en demandent aucun.

Écrivez `generale(moteur, question, avec_outils)`, le texte de la réponse du moteur à une question seule (un message `user`, sans consigne), avec les outils `OUTILS` si `avec_outils`, et le nombre de ses appels. Posez chaque question au moteur de base et au spécialisé, et rangez dans `generales` une ligne par question et par moteur : `sans_appel` (aucun appel, ni bloc `<tool_call>` dans le texte), `juste` (le motif trouvé dans le texte) et `marque` (le texte parle de Biclou). Comparez les deux moteurs, question par question. Quelle différence serait une régression ? Que faudrait-il pour la mesurer vraiment ?

**Documentation utile**

- [`re.search`](https://docs.python.org/3/library/re.html#re.search)

In [ ]:
GENERALES = [
  # (question, posée avec les outils de l'agent, le motif d'une réponse juste)
  ("Combien font 17 × 23 ? Donne le nombre seul.", False, r"\b391\b"),
  ("Quelle est la capitale de l'Australie ? Réponds en un mot.", False, r"canberra"),
  ("Combien de jours compte une année bissextile ? Le nombre seul.", False, r"\b366\b"),
  (
    "Range ces fruits dans l'ordre alphabétique : poire, abricot, melon.",
    False,
    r"abricot.*melon.*poire",
  ),
  (
    "Traduis en anglais : « Le vélo est dans le garage. »",
    False,
    r"bi(ke|cycle).*garage",
  ),
  (
    "Corrige cette phrase : « Nous avons pas reçu le colis. »",
    False,
    r"n['’]avons pas",
  ),
  ("Quelle est la capitale de l'Italie ? Réponds en un mot.", True, r"rome"),
  ("De quelle couleur est le ciel par temps clair ? Un mot.", True, r"bleu"),
]

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- La requête : `moteur.client.chat.completions.create(model=moteur.modele, messages=[dict(role="user", content=question)], **outils, **moteur.options)`, avec `outils = dict(tools=OUTILS) if avec_outils else {}`
- Sans outils dans la requête, le serveur ne lit aucun appel : un modèle qui en écrit un le laisse dans le texte, en bloc `<tool_call>`
- `re.search(motif, texte, re.I | re.S)` : `re.I` ignore la casse, `re.S` laisse `.` passer les retours à la ligne ; `None` sans correspondance
- `generales.pivot(index="question", columns="moteur", values="juste")` : une ligne par question, une colonne par moteur

```python
def generale(moteur: Moteur, question: str, avec_outils: bool) -> tuple[str, int]:
  outils = dict(tools=OUTILS) if avec_outils else {}
  reponse = moteur.client.chat.completions.create(...)
  message = reponse.choices[0].message
  return message.content or "", len(message.tool_calls or [])


lignes_generales = []
for nom in ["base", "specialise"]:
  for question, avec_outils, motif in GENERALES:
    texte, appels = generale(MOTEURS[nom], question, avec_outils)
    lignes_generales.append(
      dict(
        moteur=nom,
        question=question[:45],
        avec_outils=avec_outils,
        sans_appel=...,
        juste=bool(re.search(...)),
        marque=...,
      )
    )
generales = pd.DataFrame(lignes_generales)
print(generales.pivot(...))
generales.groupby("moteur")[["sans_appel", "juste", "marque"]].mean()
```

#### Solution

In [ ]:
def generale(moteur: Moteur, question: str, avec_outils: bool) -> tuple[str, int]:
  """La réponse du moteur à une question seule, et le nombre de ses appels."""
  outils = dict(tools=OUTILS) if avec_outils else {}
  reponse = moteur.client.chat.completions.create(
    model=moteur.modele,
    messages=[dict(role="user", content=question)],
    **outils,
    **moteur.options,
  )
  message = reponse.choices[0].message
  return message.content or "", len(message.tool_calls or [])


lignes_generales = []
for nom in ["base", "specialise"]:
  for question, avec_outils, motif in GENERALES:
    texte, appels = generale(MOTEURS[nom], question, avec_outils)
    lignes_generales.append(
      dict(
        moteur=nom,
        question=question[:45],
        avec_outils=avec_outils,
        sans_appel=appels == 0 and "<tool_call>" not in texte,
        juste=bool(re.search(motif, texte, re.I | re.S)),
        marque="biclou" in texte.lower(),
      )
    )
generales = pd.DataFrame(lignes_generales)
print(generales.pivot(index="question", columns="moteur", values="juste"), "\n")
generales.groupby("moteur")[["sans_appel", "juste", "marque"]].mean().round(2)

# Une régression : une question que le modèle de base réussit et que le spécialisé
# rate, un appel d'outil sur une question qui n'en demande pas, une réponse générale
# signée Biclou. Huit questions détectent une fuite grossière, pas une baisse de
# quelques points : celle-ci se mesure sur des jeux de référence publics (MMLU, GSM8K,
# IFEval, et leurs versions françaises), avant et après l'affinage, avec le même
# protocole. Ces huit questions restent dans le banc comme test de fumée.

## 2. L'agent de bout en bout

L'exercice 1 juge chaque étape sur l'historique correct. En production, l'agent enchaîne ses propres appels, sur les vrais outils, et peut s'égarer comme se rattraper. Faites tourner `agent_robuste` sur chaque demande de l'échantillon, pour chaque moteur, chaque fois avec un serveur MCP neuf (`async with Client(SERVEUR)` dans la boucle des demandes : ses compteurs de tickets et de dossiers repartent du début), et décidez de la réussite par du code, sans juge. Écrivez :

- `reussie(demande, trajet)` : chaque appel attendu est fait, avec ses arguments vérifiables exacts (`trouves`, qui compare comme `meme_appel` : `numero` ; `modele`, `taille` et `magasin` ; `theme` ; `categorie` et `priorite` ; `raison`), aucune écriture (`ECRITURES`) que la demande n'attend pas, et pas de réponse de repli (`REPLI`, la demande passée à un conseiller) ;
- `async banc(nom, moteur, demandes)`, une ligne par demande : le nom du moteur, l'`id`, le nombre d'appels attendus, `reussie`, puis la trajectoire : les `generations`, les `appels` exécutés, les `superflus` (ceux qui ne correspondent à aucun appel attendu), les `relances`, les `refus` (les appels que `verifie` a refusés, `appels_refuses`), `repli`, les trois comparaisons des diapositives, `stricte` (les appels exécutés sont les attendus, dans l'ordre), `sans_ordre` (les mêmes appels, dans n'importe quel ordre) et `sur_ensemble` (tous les appels attendus, d'autres tolérés), puis `erreurs_outils` ; la durée en `secondes`, sans le lancement du serveur MCP, et les tokens du trajet (`tokens_prompt`, `tokens_caches`, `tokens_generes`) ; enfin, pour les exercices suivants, les noms des appels `attendus` et `obtenus`, les `signatures` des appels exécutés (`signature`, qui compare deux appels comme `meme_appel`), le `transfert`, la `reponse` et la conversation (`messages`).

Rangez les lignes de tous les moteurs dans `resultats`, et affichez la moyenne des `COLONNES` par moteur, puis les raisons des transferts. Le taux de réussite de bout en bout est-il celui de l'exercice 1 ? Que disent les écarts entre `stricte`, `sans_ordre` et `sur_ensemble` ? Quelle part des relances vient de la validation (`refus`), un composant de l'agent, et quelle part des outils (`erreurs_outils`) ? Une trajectoire réussie peut-elle coûter plus qu'une autre ?

In [ ]:
ECRITURES = {"creer_ticket", "escalader"}
COLONNES = [
  "reussie",
  "generations",
  "appels",
  "superflus",
  "relances",
  "refus",
  "repli",
  "stricte",
  "sans_ordre",
  "sur_ensemble",
  "erreurs_outils",
  "secondes",
]


def erreurs_outils(messages: list[dict]) -> int:
  """Les résultats d'outils en erreur d'une conversation, sans les refus de
  verifie."""
  return sum(
    m["role"] == "tool"
    and '"erreur"' in m["content"]
    and "appel refusé" not in m["content"]
    for m in messages
  )


def appels_refuses(messages: list[dict]) -> int:
  """Les appels d'une conversation que verifie a refusés."""
  return sum(m["role"] == "tool" and "appel refusé" in m["content"] for m in messages)


def signature(appel: dict) -> str:
  """L'outil d'un appel et ses arguments vérifiables, en JSON trié : deux appels ont
  la même signature quand meme_appel les dit identiques."""
  return json.dumps([appel["name"], exacts(appel)], ensure_ascii=False, sort_keys=True)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Les écritures de trop : `ecritures = [a for a in trajet["appels"] if a["name"] in ECRITURES]`, et de même pour les appels attendus ; `trouves(ecritures, ecritures_attendues) == len(ecritures)` : chaque écriture faite était attendue
- `trajet["appels"]` ne garde que les appels exécutés : un appel refusé par `verifie` n'en est pas
- Une demande : `async with Client(SERVEUR) as mcp:`, puis, dedans, `debut = time.time()` et `trajet = await agent_robuste(moteur, mcp, messages)`, avec `messages = conversation(demande["message"])`, que la boucle allonge
- `stricte` : autant d'appels que d'attendus, et `meme_appel` deux à deux (`zip`) ; `sans_ordre` : tous les attendus trouvés, et autant d'appels que d'attendus ; `sur_ensemble` : tous les attendus trouvés
- `resultats.groupby("moteur")[COLONNES].mean()` ; `pd.crosstab(resultats["moteur"], resultats["transfert"].fillna("aucun"))` : les raisons des transferts
- Sur deux processeurs, une demande prend de quelques secondes à une minute avec Qwen3-0.6B

```python
def reussie(demande: dict, trajet: dict) -> bool:
  attendus, appels = demande["appels_attendus"], trajet["appels"]
  ecritures = [a for a in appels if a["name"] in ECRITURES]
  ecritures_attendues = ...
  return (
    trouves(attendus, appels) == len(attendus)
    and trouves(ecritures, ecritures_attendues) == len(ecritures)
    and ...
  )


async def banc(nom: str, moteur: Moteur, demandes: pd.DataFrame) -> list[dict]:
  lignes = []
  for demande in demandes.to_dict("records"):
    messages = conversation(demande["message"])
    async with Client(SERVEUR) as mcp:  # un serveur neuf
      debut = time.time()
      trajet = await agent_robuste(...)
      secondes = time.time() - debut
    attendus, appels = demande["appels_attendus"], trajet["appels"]
    n = trouves(attendus, appels)
    lignes.append(
      dict(
        moteur=nom,
        id=demande["id"],
        appels_attendus=len(attendus),
        reussie=reussie(demande, trajet),
        generations=trajet["generations"],
        appels=len(appels),
        superflus=...,
        relances=trajet["relances"],
        refus=appels_refuses(messages),
        repli=trajet["reponse"] == REPLI,
        stricte=...,
        sans_ordre=...,
        sur_ensemble=...,
        erreurs_outils=erreurs_outils(messages),
        secondes=secondes,
        tokens_prompt=trajet.get("tokens_prompt", 0),
        tokens_caches=...,
        tokens_generes=...,
        attendus=[a["name"] for a in attendus],
        obtenus=[a["name"] for a in appels],
        signatures=[signature(a) for a in appels],
        transfert=trajet["transfert"],
        reponse=trajet["reponse"],
        messages=messages,
      )
    )
  return lignes


lignes = []
for nom, moteur in MOTEURS.items():
  lignes += await banc(nom, moteur, echantillon)
resultats = pd.DataFrame(lignes)
print(resultats.groupby("moteur")[COLONNES].mean().round(2))
pd.crosstab(resultats["moteur"], resultats["transfert"].fillna("aucun"))
```

### Solution

In [ ]:
def reussie(demande: dict, trajet: dict) -> bool:
  """La tâche réussie : les appels attendus faits, aucune écriture de trop, pas de
  réponse de repli."""
  attendus, appels = demande["appels_attendus"], trajet["appels"]
  ecritures = [a for a in appels if a["name"] in ECRITURES]
  ecritures_attendues = [a for a in attendus if a["name"] in ECRITURES]
  return (
    trouves(attendus, appels) == len(attendus)
    and trouves(ecritures, ecritures_attendues) == len(ecritures)
    and trajet["reponse"] != REPLI
  )


async def banc(nom: str, moteur: Moteur, demandes: pd.DataFrame) -> list[dict]:
  """L'agent sur chaque demande, avec un serveur MCP neuf : une ligne par demande."""
  lignes = []
  for demande in demandes.to_dict("records"):
    messages = conversation(demande["message"])
    async with Client(SERVEUR) as mcp:  # un serveur neuf : des compteurs au début
      debut = time.time()
      trajet = await agent_robuste(moteur, mcp, messages)
      secondes = time.time() - debut
    attendus, appels = demande["appels_attendus"], trajet["appels"]
    n = trouves(attendus, appels)
    lignes.append(
      dict(
        moteur=nom,
        id=demande["id"],
        appels_attendus=len(attendus),
        reussie=reussie(demande, trajet),
        generations=trajet["generations"],
        appels=len(appels),
        superflus=len(appels) - n,
        relances=trajet["relances"],
        refus=appels_refuses(messages),
        repli=trajet["reponse"] == REPLI,
        stricte=len(appels) == len(attendus)
        and all(meme_appel(o, a) for o, a in zip(appels, attendus)),
        sans_ordre=n == len(attendus) == len(appels),
        sur_ensemble=n == len(attendus),
        erreurs_outils=erreurs_outils(messages),
        secondes=secondes,
        tokens_prompt=trajet.get("tokens_prompt", 0),
        tokens_caches=trajet.get("tokens_caches", 0),
        tokens_generes=trajet.get("tokens_generes", 0),
        attendus=[a["name"] for a in attendus],
        obtenus=[a["name"] for a in appels],
        signatures=[signature(a) for a in appels],
        transfert=trajet["transfert"],
        reponse=trajet["reponse"],
        messages=messages,
      )
    )
  return lignes


debut = time.time()
lignes = []
for nom, moteur in MOTEURS.items():
  lignes += await banc(nom, moteur, echantillon)
resultats = pd.DataFrame(lignes)
print(f"{time.time() - debut:.0f} s\n")
print(resultats.groupby("moteur")[COLONNES].mean().round(2), "\n")
pd.crosstab(resultats["moteur"], resultats["transfert"].fillna("aucun"))

# La réussite de bout en bout n'est pas le taux correct de l'exercice 1 : l'agent peut
# rater une étape et se rattraper (un appel refusé, puis corrigé), ou réussir chaque
# étape seule et s'égarer en chaîne (un appel de trop, le budget d'étapes épuisé).
# sur_ensemble sans sans_ordre : des appels en plus, une lecture de trop qu'on tolère
# ou non. sans_ordre sans stricte : les bons appels dans un autre ordre, sans
# conséquence pour deux lectures indépendantes, pas quand le ticket doit citer la
# commande lue avant lui. Les relances sont les refus plus les erreurs d'outils. Les
# refus mesurent un composant, la validation (verifie), qui arrête un numéro inventé
# ou un argument hors du schéma ; les erreurs d'outils, un appel valide que le système
# ne sait pas servir (une commande introuvable). Ces colonnes disent ce que la
# validation a refusé, pas ce qu'elle a laissé passer : l'évaluer seule demande son
# propre jeu, des appels invalides écrits à la main, tous à refuser. Deux trajectoires
# réussies ne coûtent pas pareil : chaque appel superflu, chaque relance coûte une
# génération et allonge le prompt ; et quatre générations ne laissent qu'une relance à
# une demande à deux appels. Les raisons des transferts disent aussi un moteur mal
# configuré : une option qu'une API refuse donne un transfert à chaque demande.

## 3. Un LLM juge pour les réponses

La réussite de l'exercice 2 juge les appels ; la réponse au client, du texte libre, ne se compare pas mot à mot à la référence. Un LLM la juge, sur une grille de quatre critères oui ou non, `CRITERES` (ci-dessous) : un critère binaire et précis se juge de façon plus fiable qu'une note de 1 à 10. Le juge est le généraliste, sans échantillonnage quand le modèle l'accepte, et rend un objet JSON au schéma de `Grille` (`FORMAT_GRILLE`, une sortie structurée).

Écrivez `juge(demande, ligne)`, la grille du juge pour la réponse d'une ligne de `resultats` : `CONSIGNES_JUGE` en message `system` ; dans le message `user`, le message du client, les résultats des outils de la conversation de la ligne (`resultats_outils`), la réponse de référence et la réponse à juger ; la grille validée par `Grille`, en dictionnaire, ou `None` si la sortie n'en est pas une ou si le généraliste ne répond pas (`openai.APIError`). Jugez les réponses de `a_juger` (ci-dessous : celles de l'exercice 2, sauf les réponses de repli, toutes identiques ; sans clé, `N_JUGE_LOCAL` par moteur), et rangez dans `jugements` une ligne par réponse : le moteur, l'`id`, un score par critère, et les décisions des deux scripts ci-dessous.

Puis calibrez le juge : un juge se vérifie là où un script sait. `script_faits` décide si les numéros et les délais en jours que cite la réponse figurent dans le message, les résultats des outils ou la référence (`None` quand elle n'en cite aucun) ; `script_ton`, si la salutation et la signature sont celles de la charte, dans la langue du client. Pour chacun, comparez ses décisions à celles du juge sur le même critère (`pd.crosstab`), là où il décide, et donnez leur taux d'accord. Puis la moyenne des scores par moteur. Quels biais connus d'un juge pèsent sur ces scores ?

**Documentation utile**

- [Les sorties structurées](https://platform.openai.com/docs/guides/structured-outputs)
- [`BaseModel.model_validate_json`](https://docs.pydantic.dev/latest/api/base_model/#pydantic.BaseModel.model_validate_json)
- [`pandas.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)

In [ ]:
CRITERES = {
  "faits": "les faits cités (numéros, dates, montants, délais, stock, règles) sont "
  "ceux des résultats des outils ou de la réponse de référence : aucun n'est inventé "
  "ni contredit",
  "suite": "la réponse donne au client une prochaine étape claire : ce qu'il doit "
  "faire, ou ce que l'équipe fait",
  "promesse": "la réponse ne promet rien que la politique ne permet : ni "
  "remboursement avant vérification, ni geste commercial, ni délai ou action que ni "
  "les outils ni la référence ne donnent ; elle ne demande ni coordonnées bancaires "
  "ni mot de passe",
  "ton": "la réponse est dans la langue du client, le vouvoie et suit la charte : "
  "« Bonjour » (« Hello » en anglais), des phrases polies, « L'équipe Biclou & Cie » "
  "(« The Biclou & Cie team ») pour signature",
}


class Grille(BaseModel):
  """La grille du juge : un oui ou un non par critère."""

  model_config = ConfigDict(extra="forbid")  # "additionalProperties": false
  faits: bool
  suite: bool
  promesse: bool
  ton: bool


FORMAT_GRILLE = {
  "type": "json_schema",
  "json_schema": {
    "name": "grille",
    "schema": Grille.model_json_schema(),
    "strict": True,
  },
}
CONSIGNES_JUGE = (
  "Tu évalues la réponse de l'agent du service client de Biclou & Cie, qui vend, "
  "répare et loue des vélos, à un message de client. Tu reçois le message, les "
  "résultats des outils que l'agent a appelés, une réponse de référence écrite pour "
  "la même demande (ses numéros de ticket et de dossier peuvent différer : ceux des "
  "outils de l'agent font foi) et la réponse à évaluer. Pour chaque critère, réponds "
  "true s'il est rempli, false sinon, sans tenir compte de la longueur de la "
  "réponse.\n" + "\n".join(f"- {nom} : {texte}" for nom, texte in CRITERES.items())
)
CITES = re.compile(r"CMD-\d{6}|T-\d+|ESC-\d+|\d+ jours")


def normalise(texte: str) -> str:
  """Sans espaces insécables ni apostrophes typographiques."""
  return texte.replace(" ", " ").replace("\xa0", " ").replace("’", "'")


def resultats_outils(ligne: dict) -> list[dict]:
  """Les résultats des outils de la conversation d'une ligne de resultats."""
  return [json.loads(m["content"]) for m in ligne["messages"] if m["role"] == "tool"]


def script_faits(demande: dict, ligne: dict) -> bool | None:
  """Les numéros et les délais en jours que cite la réponse figurent-ils dans le
  message, les résultats des outils ou la référence ? None si elle n'en cite aucun."""
  cites = set(CITES.findall(normalise(ligne["reponse"])))
  if not cites:
    return None
  outils = json.dumps(resultats_outils(ligne), ensure_ascii=False)
  sources = normalise(demande["message"] + outils + demande["reponse"])
  return all(cite in sources for cite in cites)


def script_ton(demande: dict, ligne: dict) -> bool:
  """La salutation et la signature de la charte, dans la langue du client."""
  reponse = normalise(ligne["reponse"])
  if demande["langue"] == "en":
    return reponse.startswith("Hello") and reponse.endswith("The Biclou & Cie team")
  return reponse.startswith("Bonjour") and reponse.endswith("L'équipe Biclou & Cie")


par_id = eval_fige.set_index("id", drop=False)  # par_id.loc[id] : une demande
a_juger = resultats[resultats["reponse"] != REPLI]
if not PAR_API:  # le juge servi ici : quelques réponses par moteur, d'abord celles
  # qui citent un numéro ou un délai, que script_faits sait juger
  cite = a_juger["reponse"].map(
    lambda texte: CITES.search(normalise(texte)) is not None
  )
  a_juger = a_juger.loc[cite.sort_values(ascending=False, kind="stable").index]
  a_juger = a_juger.groupby("moteur").head(N_JUGE_LOCAL)
print(len(a_juger), "réponses à juger :", a_juger["moteur"].value_counts().to_dict())

In [ ]:
# Votre code ici

### Conseils supplémentaires

- La requête : `GENERALISTE.client.chat.completions.create(model=GENERALISTE.modele, messages=..., response_format=FORMAT_GRILLE, **GENERALISTE.options)`
- Le message `user`, par exemple : `f"Message du client :\n{demande['message']}\n\nRésultats des outils de l'agent :\n{json.dumps(resultats_outils(ligne), ensure_ascii=False)}\n\nRéponse de référence :\n{demande['reponse']}\n\nRéponse à évaluer :\n{ligne['reponse']}"`
- `Grille.model_validate_json(texte).model_dump()` : la grille en dictionnaire, ou `ValidationError` pour un texte qui n'en est pas une
- Une ligne de `jugements` sans grille garde ses colonnes : `**(grille or dict.fromkeys(CRITERES))` ; `.astype(float)` avant une moyenne, sur des colonnes où manquent des valeurs
- L'accord : sur les lignes où le script décide (`jugements["script_faits"].notna()`) et où le juge a rendu une grille, `(sur["faits"] == sur["script_faits"]).mean()`
- Sans clé, sur deux processeurs, le juge local prend une à deux minutes par réponse

```python
def juge(demande: dict, ligne: dict) -> dict | None:
  contenu = ...
  try:
    reponse = GENERALISTE.client.chat.completions.create(...)
    return Grille.model_validate_json(
      reponse.choices[0].message.content or ""
    ).model_dump()
  except (openai.APIError, ValidationError):
    return None


lignes_jugees = []
for ligne in a_juger.to_dict("records"):
  demande = par_id.loc[ligne["id"]].to_dict()
  grille = juge(demande, ligne)
  lignes_jugees.append(
    dict(
      moteur=ligne["moteur"],
      id=ligne["id"],
      grille=grille is not None,
      **(grille or dict.fromkeys(CRITERES)),
      script_faits=script_faits(demande, ligne),
      script_ton=...,
    )
  )
jugements = pd.DataFrame(lignes_jugees)
for critere in ["faits", "ton"]:
  sur = jugements[jugements[f"script_{critere}"].notna() & jugements["grille"]]
  print(pd.crosstab(sur[f"script_{critere}"], sur[critere]))
  print("Accord :", ...)
jugements.astype({c: float for c in CRITERES}).groupby("moteur")[list(CRITERES)].mean()
```

### Solution

In [ ]:
def juge(demande: dict, ligne: dict) -> dict | None:
  """La grille du juge pour la réponse d'une ligne, ou None."""
  contenu = (
    f"Message du client :\n{demande['message']}\n\n"
    "Résultats des outils de l'agent :\n"
    f"{json.dumps(resultats_outils(ligne), ensure_ascii=False)}\n\n"
    f"Réponse de référence :\n{demande['reponse']}\n\n"
    f"Réponse à évaluer :\n{ligne['reponse']}"
  )
  messages = [
    dict(role="system", content=CONSIGNES_JUGE),
    dict(role="user", content=contenu),
  ]
  try:
    reponse = GENERALISTE.client.chat.completions.create(
      model=GENERALISTE.modele,
      messages=messages,
      response_format=FORMAT_GRILLE,
      **GENERALISTE.options,
    )
    texte = reponse.choices[0].message.content or ""
    return Grille.model_validate_json(texte).model_dump()
  except (openai.APIError, ValidationError):
    return None


debut = time.time()
lignes_jugees = []
for ligne in a_juger.to_dict("records"):
  demande = par_id.loc[ligne["id"]].to_dict()
  grille = juge(demande, ligne)
  lignes_jugees.append(
    dict(
      moteur=ligne["moteur"],
      id=ligne["id"],
      grille=grille is not None,
      **(grille or dict.fromkeys(CRITERES)),
      script_faits=script_faits(demande, ligne),
      script_ton=script_ton(demande, ligne),
    )
  )
jugements = pd.DataFrame(lignes_jugees)
print(f"{len(jugements)} réponses jugées en {time.time() - debut:.0f} s")
print("Grilles valides :", jugements["grille"].sum())
for critere in ["faits", "ton"]:
  sur = jugements[jugements[f"script_{critere}"].notna() & jugements["grille"]]
  if sur.empty:
    print(f"\n{critere} : le script ne décide d'aucune réponse jugée")
    continue
  print(f"\n{critere}, le script en lignes, le juge en colonnes :")
  print(pd.crosstab(sur[f"script_{critere}"], sur[critere]))
  accord = (sur[critere] == sur[f"script_{critere}"]).mean()
  print(f"Accord sur {len(sur)} réponses : {accord:.2f}")
scores = jugements.astype({c: float for c in CRITERES})
scores.groupby("moteur")[list(CRITERES)].mean().round(2)

# Là où le script dit non (un numéro absent de tout ce que l'agent a lu, une signature
# manquante), le juge doit dire non ; là où il dit oui, le juge peut dire non pour une
# raison que le script ne voit pas (le registre, un fait faux sans chiffre). Un juge
# qui dit oui là où le script dit non est complaisant : un petit modèle répond souvent
# oui. Les biais connus : la longueur (une réponse longue paraît meilleure), la
# préférence pour sa propre famille (ici, le généraliste juge aussi ses propres
# réponses), la référence, qui ancre le juge sur une formulation. On s'en garde par
# des critères binaires et précis, un juge d'une autre famille, et un échantillon noté
# à la main, sur lequel on mesure l'accord avant de croire les scores. Avec si peu de
# réponses, l'accord se lit cas par cas, pas en taux.

## 4. Coût et latence

À qualité proche, le modèle retenu est celui qui coûte le moins et répond assez vite. L'exercice 2 a gardé, pour chaque demande, sa durée et ses tokens : ceux du prompt, ceux que le serveur a repris de son cache (`tokens_caches` : un serveur local s'épargne leur calcul, un fournisseur les facture moins cher), ceux qu'il a générés. Calculez, par moteur :

- la latence d'une demande, sa médiane (p50) et son 95e centile (p95), en secondes ;
- les tokens moyens d'une demande, du prompt, du cache et de la génération ;
- le coût de `VOLUME` demandes par mois. Pour le généraliste, ses tokens aux prix de `PRIX_API` (ci-dessous), en dollars par million de tokens, ceux du cache au prix du cache. Pour un moteur servi ici, des machines louées tout le mois (`HEURES_MOIS` heures, payées même quand elles attendent), autant qu'il en faut pour servir `VOLUME` demandes à son débit mesuré (une demande à la fois : `3600 / secondes` demandes par heure), au prix de l'heure de chaque machine de `PRIX_HEURE`.

Puis le volume mensuel au-delà duquel un GPU loué tout le mois revient moins cher que le généraliste par API.

Les prix sont des constantes datées : remplacez-les par ceux du jour de votre fournisseur. Le débit mesuré est celui de la machine de ce notebook, qui sert une demande à la fois : il ne dit rien d'un serveur GPU de production, qui en sert plusieurs à la fois (vLLM). Le nombre de machines qu'il en déduit est un majorant grossier, que seule une mesure sur la machine de production remplace, et le volume de bascule suppose qu'un GPU suffit à le servir. Sans clé, estimez le coût du généraliste sur les tokens du spécialisé, qui appelle ses outils comme le ferait le généraliste.

In [ ]:
VOLUME = 10_000  # demandes par mois
# Les prix publics de gpt-5.4-mini relevés le 2026-11-18, en dollars par million de
# tokens : l'entrée, l'entrée lue dans le cache, la sortie. À remplacer par ceux du
# jour.
PRIX_API = {"entree": 0.75, "cache": 0.075, "sortie": 4.50}
# Le prix de l'heure d'une machine louée, en dollars : deux processeurs, comme ceux de
# ce notebook, et un GPU L4. Des ordres de grandeur de 2025, à remplacer.
PRIX_HEURE = {"cpu": 0.10, "gpu": 0.80}
HEURES_MOIS = 730  # les heures d'un mois, qu'une machine louée au mois se paie

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `resultats.groupby("moteur")["secondes"].quantile([0.5, 0.95]).unstack()` : une colonne par quantile
- Le coût d'une demande par API : `((tokens_prompt - tokens_caches) * PRIX_API["entree"] + tokens_caches * PRIX_API["cache"] + tokens_generes * PRIX_API["sortie"]) / 1e6` dollars ; un modèle de raisonnement facture aussi ses tokens de raisonnement, comptés dans `completion_tokens`
- Les machines d'un moteur servi ici : `heures = VOLUME * lignes["secondes"].mean() / 3600` d'occupation, donc `max(1, math.ceil(heures / HEURES_MOIS))` machines louées tout le mois, chacune `HEURES_MOIS * prix`
- Le volume de bascule : le prix d'un GPU au mois, `HEURES_MOIS * PRIX_HEURE["gpu"]`, divisé par le coût d'une demande par API
- `pd.DataFrame(couts).T` : un dictionnaire de dictionnaires en tableau, une ligne par moteur
- Exemple à vérifier à la main : 1 500 tokens de prompt dont 1 300 en cache et 100 générés coûtent (200 × 0,75 + 1 300 × 0,075 + 100 × 4,50) / 10⁶ = 0,0006975 dollar, près de 7 dollars les 10 000 ; un GPU à 0,80 dollar l'heure, 584 dollars le mois, revient moins cher au-delà de 837 000 demandes par mois environ

```python
def cout_api(lignes: pd.DataFrame) -> float:
  non_caches = lignes["tokens_prompt"] - lignes["tokens_caches"]
  dollars = (
    non_caches * PRIX_API["entree"] + lignes["tokens_caches"] * PRIX_API["cache"] + ...
  ) / 1e6
  return dollars.mean()


latence = resultats.groupby("moteur")["secondes"].quantile([0.5, 0.95]).unstack()
latence.columns = ["p50", "p95"]
colonnes_tokens = ["tokens_prompt", "tokens_caches", "tokens_generes"]
print(latence.round(1))
print(resultats.groupby("moteur")[colonnes_tokens].mean().round(0))
couts = {}
for nom, lignes in resultats.groupby("moteur"):
  if nom == "generaliste":
    couts[nom] = {"api": VOLUME * cout_api(lignes)}
  else:
    heures = ...
    machines = ...
    couts[nom] = {m: machines * HEURES_MOIS * prix for m, prix in PRIX_HEURE.items()}
par_api = resultats[resultats["moteur"] == ("generaliste" if PAR_API else "specialise")]
if not PAR_API:
  couts["generaliste (estimé)"] = {"api": ...}
couts = pd.DataFrame(couts).T
print(couts.round(2))
gpu_mois = ...
print("Un GPU au mois revient moins cher au-delà de", ..., "demandes par mois")
```

### Solution

In [ ]:
def cout_api(lignes: pd.DataFrame) -> float:
  """Le coût moyen d'une demande aux prix de PRIX_API, en dollars."""
  non_caches = lignes["tokens_prompt"] - lignes["tokens_caches"]
  dollars = (
    non_caches * PRIX_API["entree"]
    + lignes["tokens_caches"] * PRIX_API["cache"]
    + lignes["tokens_generes"] * PRIX_API["sortie"]
  ) / 1e6
  return dollars.mean()


latence = resultats.groupby("moteur")["secondes"].quantile([0.5, 0.95]).unstack()
latence.columns = ["p50", "p95"]
colonnes_tokens = ["tokens_prompt", "tokens_caches", "tokens_generes"]
print("Latence d'une demande, en secondes :")
print(latence.round(1), "\n")
print("Tokens moyens d'une demande :")
print(resultats.groupby("moteur")[colonnes_tokens].mean().round(0), "\n")
couts = {}
for nom, lignes in resultats.groupby("moteur"):
  if nom == "generaliste":
    couts[nom] = {"api": VOLUME * cout_api(lignes)}
  else:  # servi ici : des machines louées tout le mois, autant que le débit l'exige
    heures = VOLUME * lignes["secondes"].mean() / 3600
    machines = max(1, math.ceil(heures / HEURES_MOIS))
    couts[nom] = {m: machines * HEURES_MOIS * prix for m, prix in PRIX_HEURE.items()}
# Le généraliste par API, ou, sans clé, estimé sur les tokens du spécialisé.
par_api = resultats[resultats["moteur"] == ("generaliste" if PAR_API else "specialise")]
if not PAR_API:
  couts["generaliste (estimé)"] = {"api": VOLUME * cout_api(par_api)}
couts = pd.DataFrame(couts).T
print(f"Coût de {VOLUME} demandes par mois, en dollars :")
print(couts.round(2), "\n")
gpu_mois = HEURES_MOIS * PRIX_HEURE["gpu"]
bascule = round(gpu_mois / cout_api(par_api), -3)
print(f"Un GPU loué tout le mois, {gpu_mois:.0f} dollars, revient moins cher que l'API")
print(f"au-delà de {bascule:,.0f} demandes par mois".replace(",", " "))

# Le p95 dit l'attente des clients les moins bien servis : la moyenne la cache. Sur
# un échantillon de cette taille, le p95 n'est guère que la demande la plus lente.
# Le cache fait l'essentiel du prompt (la consigne et les schémas des outils) : sans
# lui, chaque demande paierait ces tokens au prix plein, ou les recalculerait. Le
# coût par API suit le volume, sans rien à installer ; une machine louée se paie tout
# le mois, même quand elle attend, et une équipe la fait tourner. Aux prix de ce
# notebook, un GPU loué pour 10 000 demandes par mois coûte bien plus que l'API : il
# ne revient moins cher qu'au-delà du volume de bascule, s'il suffit à le servir. Le
# débit de ce notebook n'est pas celui d'un serveur GPU qui sert plusieurs demandes à
# la fois : seule une mesure sur la machine de production, à la charge prévue, dit
# combien de machines il faut.

## 5. Analyse d'erreurs et recommandation

### 5.1 À qui la faute ?

Un taux d'échec ne dit pas quoi corriger. Écrivez `cause(ligne)`, la cause d'un échec de l'exercice 2 (une ligne de `resultats` où `reussie` est faux), la première qui s'applique, dans cet ordre :

1. `attendu` : tous les moteurs ratent la demande, avec les mêmes appels, arguments vérifiables compris (`signatures`, au moins un appel) ; quand les modèles s'accordent contre l'étiquette, c'est elle qu'il faut relire ;
2. `outils` : un outil a renvoyé une erreur pendant le trajet (`erreurs_outils`), ou le généraliste rate la même demande avec les mêmes appels (`signatures`, au moins un) : un modèle plus fort trompé de la même façon désigne la description ou le schéma d'un outil ;
3. `modele` : le moteur rate aussi la demande à l'exercice 1 (`pas`), sur l'historique correct : il ne sait pas faire ;
4. `enchainement` : le moteur réussit chaque étape seule, mais rate l'enchaînement : le budget d'étapes, l'historique de ses propres appels, le gabarit, la boucle.

Rangez les échecs dans `echecs`, avec leur `cause`, et comptez-les par moteur et par cause. Puis lisez cinq échecs, de moteurs et de causes différents autant que possible : le message, les appels attendus et obtenus, le transfert, le début de la réponse. La règle trie ; seule la lecture confirme une cause.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Les lignes d'une même demande : `resultats[resultats["id"] == ligne["id"]]` ; ses échecs, `memes[~memes["reussie"]]`
- Les mêmes appels, arguments vérifiables compris : `ratees["signatures"].map(tuple).nunique() == 1` (une liste ne se compare pas dans un ensemble, un tuple si) ; `obtenus`, les seuls noms, confondrait deux `chercher_politique` de thèmes différents
- L'exercice 1 pour ce moteur et cette demande : `pas[(pas["moteur"] == ligne["moteur"]) & (pas["id"] == ligne["id"])]["correct"]`
- `echecs.drop_duplicates(["moteur", "cause"])` : un échec par moteur et par cause, de quoi choisir des échecs variés

```python
def cause(ligne: dict) -> str:
  memes = resultats[resultats["id"] == ligne["id"]]
  ratees = memes[~memes["reussie"]]
  if (
    len(ratees) == len(MOTEURS)
    and ratees["signatures"].map(tuple).nunique() == 1
    and ligne["signatures"]
  ):
    return "attendu"
  generaliste = ratees[ratees["moteur"] == "generaliste"]
  pareil = ligne["moteur"] != "generaliste" and ligne["signatures"] and ...
  if ligne["erreurs_outils"] > 0 or pareil:
    return "outils"
  etape = pas[...]
  if not etape["correct"].all():
    return ...
  return "enchainement"


echecs = resultats[~resultats["reussie"]].copy()
echecs["cause"] = [cause(ligne) for ligne in echecs.to_dict("records")]
print(pd.crosstab(echecs["moteur"], echecs["cause"]))
a_lire = pd.concat([echecs.drop_duplicates(["moteur", "cause"]), echecs])
for ligne in a_lire.drop_duplicates(["moteur", "id"]).head(5).to_dict("records"):
  demande = par_id.loc[ligne["id"]]
  print(f"\n=== {ligne['moteur']}, {ligne['id']} : {ligne['cause']}")
  print(...)
```

#### Solution

In [ ]:
def cause(ligne: dict) -> str:
  """La cause d'un échec de l'exercice 2, la première qui s'applique."""
  memes = resultats[resultats["id"] == ligne["id"]]
  ratees = memes[~memes["reussie"]]
  if (
    len(ratees) == len(MOTEURS)
    and ratees["signatures"].map(tuple).nunique() == 1
    and ligne["signatures"]
  ):
    return "attendu"
  generaliste = ratees[ratees["moteur"] == "generaliste"]
  pareil = (
    ligne["moteur"] != "generaliste"
    and ligne["signatures"]
    and any(s == ligne["signatures"] for s in generaliste["signatures"])
  )
  if ligne["erreurs_outils"] > 0 or pareil:
    return "outils"
  etape = pas[(pas["moteur"] == ligne["moteur"]) & (pas["id"] == ligne["id"])]
  if not etape["correct"].all():
    return "modele"
  return "enchainement"


echecs = resultats[~resultats["reussie"]].copy()
echecs["cause"] = [cause(ligne) for ligne in echecs.to_dict("records")]
print(pd.crosstab(echecs["moteur"], echecs["cause"]))
a_lire = pd.concat([echecs.drop_duplicates(["moteur", "cause"]), echecs])
for ligne in a_lire.drop_duplicates(["moteur", "id"]).head(5).to_dict("records"):
  demande = par_id.loc[ligne["id"]]
  print(
    f"\n=== {ligne['moteur']}, {ligne['id']} ({demande['scenario']}) : {ligne['cause']}"
  )
  print("Message :", " ".join(demande["message"].split())[:200])
  print("Attendus :", [(a["name"], exacts(a)) for a in demande["appels_attendus"]])
  print("Obtenus :", ligne["obtenus"], "| transfert :", ligne["transfert"])
  print("Réponse :", " ".join(ligne["reponse"].split())[:150])

# Ce que chaque cause demande : modele, des exemples d'entraînement de ce cas (le
# chapitre 3) ; enchainement, la boucle (le budget d'étapes, ce que l'historique garde)
# ou le gabarit servi ; outils, la description ou le schéma de l'outil, une erreur plus
# claire ; attendu, relire l'étiquette, la corriger s'il le faut, et le noter : le jeu
# figé change de version. La règle se trompe : un appel inutile qui précède le bon
# outil, une erreur d'outil provoquée par un argument faux du modèle. Lire les échecs
# tranche, et la lecture fait souvent apparaître une cause que la règle n'avait pas.

### 5.2 La recommandation

Remplissez le banc : `bilan`, une ligne par moteur, avec en colonnes les sept lignes du tableau des diapositives : les appels corrects pas à pas (`correct`, exercice 1.1), aucun appel quand aucun n'est attendu (le même taux, sur les demandes sans appel attendu), la réussite de bout en bout (exercice 2), le critère `faits` du juge (exercice 3), la non-régression (`juste`, exercice 1.2), la latence p95 et le coût de `VOLUME` demandes par mois (exercice 4). Ajoutez à la réussite sa marge, l'intervalle de Wilson à 95 % sur les `N_ECHANTILLON` demandes de l'échantillon.

Puis appliquez la règle de décision, fixée avant le banc : parmi les moteurs qui tiennent les exigences de la cellule de configuration (`SEUIL_REUSSITE`, `P95_MAX`), ceux que l'intervalle de Wilson ne départage pas du meilleur d'entre eux (leurs intervalles se recouvrent), et parmi ceux-là, le moins cher au volume prévu.

Enfin, écrivez dans la cellule de texte votre recommandation à l'équipe : le modèle à mettre en production, et pourquoi, les chiffres du banc à l'appui ; ce qu'il reste à mesurer, ou à corriger, avant de décider.

**Documentation utile**

- [L'intervalle de Wilson](https://en.wikipedia.org/wiki/Binomial_proportion_confidence_interval#Wilson_score_interval)

In [ ]:
# Votre code ici

*Votre recommandation ici.*

#### Conseils supplémentaires

- L'intervalle de Wilson de $k$ réussites sur $n$ demandes, avec $z = 1{,}96$ : un centre $\frac{k + z^2/2}{n + z^2}$, une demi-largeur $\frac{z}{n + z^2}\sqrt{\frac{k(n-k)}{n} + \frac{z^2}{4}}$ ; exemple à vérifier : 8 réussites sur 12 donnent [0,39 ; 0,86]
- Sur un petit échantillon, il est plus juste que l'intervalle « p ± 1,96 écart-type », qui sort de [0 ; 1] près des bords
- Les lignes du banc : `pas.groupby("moteur")["correct"].mean()`, et de même sur `pas[pas["appels_attendus"] == 0]` ; `scores.groupby("moteur")["faits"].mean()` (exercice 3) ; `generales.groupby("moteur")["juste"].mean()` (exercice 1.2) ; `latence["p95"]` ; `couts.min(axis=1)`, la machine la moins chère. Une valeur qu'un exercice n'a pas mesurée reste vide (`NaN`) : le généraliste n'a pas de non-régression
- `bilan[LIGNES].T` : le banc tel que dans les diapositives, une colonne par moteur
- La règle : `(bilan["reussite"] >= SEUIL_REUSSITE) & (bilan["p95"] <= P95_MAX)`, les moteurs qui tiennent les exigences ; parmi eux, le meilleur taux, et ceux dont la borne haute atteint sa borne basse ; parmi ceux-là, le moins cher. Si aucun moteur ne tient les exigences, c'est aussi une recommandation

```python
def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
  centre = (k + z**2 / 2) / (n + z**2)
  marge = ...
  return centre - marge, centre + marge


LIGNES = [
  "appels_corrects",
  "aucun_appel",
  "reussite",
  "faits",
  "non_regression",
  "p95",
  "cout",
]
bilan = resultats.groupby("moteur")["reussie"].agg(reussies="sum", n="size")
bilan["appels_corrects"] = pas.groupby("moteur")["correct"].mean()
sans_appel = pas[pas["appels_attendus"] == 0]
bilan["aucun_appel"] = ...
bilan["reussite"] = bilan["reussies"] / bilan["n"]
bilan[["bas", "haut"]] = [wilson(k, n) for k, n in zip(bilan["reussies"], bilan["n"])]
bilan["faits"] = ...
bilan["non_regression"] = ...
bilan["p95"] = latence["p95"]
bilan["cout"] = couts.min(axis=1)
print(bilan[LIGNES].T.round(2))
candidats = bilan[...]
if candidats.empty:
  print("Aucun moteur ne tient les exigences")
else:
  meilleur = candidats["reussite"].idxmax()
  pas_departages = candidats[candidats["haut"] >= ...]
  print("Le moins cher d'entre eux :", pas_departages["cout"].idxmin())
```

#### Solution

In [ ]:
def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
  """L'intervalle de Wilson d'une proportion, k réussites sur n, à 95 %."""
  centre = (k + z**2 / 2) / (n + z**2)
  marge = z / (n + z**2) * math.sqrt(k * (n - k) / n + z**2 / 4)
  return centre - marge, centre + marge


# Les sept lignes du banc des diapositives, dans leur ordre.
LIGNES = [
  "appels_corrects",
  "aucun_appel",
  "reussite",
  "faits",
  "non_regression",
  "p95",
  "cout",
]
bilan = resultats.groupby("moteur")["reussie"].agg(reussies="sum", n="size")
bilan["appels_corrects"] = pas.groupby("moteur")["correct"].mean()
sans_appel = pas[pas["appels_attendus"] == 0]
bilan["aucun_appel"] = sans_appel.groupby("moteur")["correct"].mean()
bilan["reussite"] = bilan["reussies"] / bilan["n"]
bilan[["bas", "haut"]] = [wilson(k, n) for k, n in zip(bilan["reussies"], bilan["n"])]
bilan["faits"] = scores.groupby("moteur")["faits"].mean()
bilan["non_regression"] = generales.groupby("moteur")["juste"].mean()
bilan["p95"] = latence["p95"]
bilan["cout"] = couts.min(axis=1)  # en dollars par mois, la machine la moins chère
print("Le banc :")
print(bilan[LIGNES].T.round(2), "\n")
print("La réussite et son intervalle de Wilson à 95 % :")
print(bilan[["reussies", "n", "bas", "haut"]].round(2), "\n")
tient = (bilan["reussite"] >= SEUIL_REUSSITE) & (bilan["p95"] <= P95_MAX)
candidats = bilan[tient]
print(f"Tiennent {SEUIL_REUSSITE * 100:.0f} % de réussite et {P95_MAX} s au p95 :")
print(list(candidats.index))
if candidats.empty:
  print("Aucun moteur ne tient les exigences : aucun à mettre en production en l'état")
else:
  meilleur = candidats["reussite"].idxmax()
  pas_departages = candidats[candidats["haut"] >= candidats.loc[meilleur, "bas"]]
  print("Le meilleur d'entre eux :", meilleur)
  print("Pas départagés de lui :", list(pas_departages.index))
  print("Le moins cher d'entre eux :", pas_departages["cout"].idxmin())
for n in [len(echantillon), len(eval_fige)]:
  bas, haut = wilson(n // 2, n)
  print(f"Marge à 50 % de réussite sur {n} demandes : ± {(haut - bas) / 2:.2f}")

Une recommandation argumentée suit cet ordre, les chiffres du banc à l'appui.

1. **Les exigences.** Les moteurs qui tiennent le taux de réussite et la latence p95 fixés avant le banc (`SEUIL_REUSSITE`, `P95_MAX`) : fixées après, on les choisirait pour le moteur qu'on préfère. La latence d'un moteur servi ici est celle de la machine de ce notebook : sur deux processeurs, un modèle local dépasse souvent `P95_MAX` ; seul un banc sur la machine de production dit s'il le tient.
2. **La qualité.** Le taux de réussite de bout en bout de chaque moteur, avec son intervalle. Sur un échantillon de douze demandes, la marge dépasse vingt points : les intervalles se recouvrent presque toujours, et le banc ne départage que des moteurs très éloignés, comme un modèle de base qui n'appelle pas ses outils. Sur les 139 demandes du jeu figé, elle tombe vers huit points. Les autres lignes du banc complètent : les appels pas à pas disent où un moteur décroche, la non-régression ce que l'affinage a coûté, le juge la réponse elle-même, dans la mesure où sa calibration le permet.
3. **Le coût**, parmi les moteurs que l'intervalle ne départage pas du meilleur : le moins cher au volume prévu. Un modèle servi par soi-même coûte ses machines, louées au mois même quand elles attendent, et une équipe ; un modèle par API, un coût qui suit le volume : la comparaison dépend du trafic prévu (le volume de bascule de l'exercice 4), et d'un débit mesuré sur la machine de production.
4. **Les réserves.** Ce que l'analyse d'erreurs a désigné à corriger avant de décider (une description d'outil, une étiquette du jeu figé), ce qui reste à mesurer (le banc sur tout le jeu figé, sur GPU ; la non-régression sur des jeux de référence), et ce qu'il faudra suivre en production : la part de transferts, de refus et de relances, qui signale un modèle qui dérive.

Une conclusion typique : retenir le spécialisé s'il tient les exigences, n'est pas départagé du généraliste et coûte moins cher au volume prévu, ce qui suppose un trafic au-delà du volume de bascule ; sinon le généraliste, ou l'architecture mixte du chapitre 5 pour les demandes que le spécialisé rate, si l'analyse d'erreurs montre qu'elles forment une famille qu'un routeur sait reconnaître.

## Arrêter les serveurs

Les serveurs tournent jusqu'à la fin de la session : arrêtez-les quand vous avez fini.

In [ ]:
for nom in list(SERVEURS):
  arrete(nom)
print("Serveurs arrêtés")

## À retenir

- Un agent s'évalue à trois niveaux : le modèle, étape par étape sur l'historique correct (ses appels, leurs arguments, et ce que l'affinage lui a fait désapprendre) ; ses composants (le routeur, la validation, chaque outil), chacun sur son propre jeu ; l'agent de bout en bout, sur ses propres appels et les vrais outils, avec sa trajectoire. Ce TP mesure le premier et le dernier, et n'aperçoit un composant, la validation, qu'à travers ses refus.
- La réussite d'une tâche se décide par du code partout où c'est possible : les appels attendus, leurs arguments vérifiables, aucune écriture de trop, pas de transfert.
- Un LLM juge note le texte libre, sur une grille de critères binaires ; il se calibre là où un script sait, et ses biais (la longueur, sa propre famille, la complaisance) se mesurent avant qu'on croie ses scores.
- Le coût se compare au volume prévu : des tokens au prix du fournisseur, cache compris, pour une API ; des machines louées au mois, payées même quand elles attendent, en nombre que fixe un débit mesuré sur le matériel de production, pour un modèle servi par soi-même. La latence se lit au p95, pas en moyenne.
- Chaque échec s'attribue au modèle, à l'enchaînement, aux outils ou à l'étiquette, puis se lit : chaque cause a sa correction. La recommandation suit une règle fixée avant le banc : les exigences de réussite et de latence, puis la qualité avec sa marge, puis le coût au volume prévu ; un échantillon sert à écrire le banc, le jeu entier à décider.